# Email Recruiter Pipeline
```
raw_emails  →  staging (auto-categorized + review)  →  recruiters (cold outreach)
```
**Categories:** `unknown` | `recruiter` | `marketing` | `newsletter` | `notification`

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import imaplib
import email
import sqlite3
import pandas as pd
from email.utils import parseaddr, parsedate_to_datetime
from dotenv import find_dotenv, load_dotenv
import os

load_dotenv(find_dotenv())
print('Libraries loaded ✓')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
YAHOO_EMAIL = 'brandoncdedwards@yahoo.com'
YMAIL_PW    = os.getenv('ymail_password')   # loaded from .env — never printed
DB_PATH     = 'emails.db'

# ── Categorization rules ──────────────────────────────────────────────────────

# Senders with more than this many emails are almost certainly marketing
MARKETING_THRESHOLD = 20

# Substrings in the local part of the email (before @) that signal marketing/automation
NOREPLY_PATTERNS = {
    'noreply', 'no-reply', 'donotreply', 'do-not-reply',
    'notifications', 'newsletter', 'updates', 'alerts',
    'mailer', 'bounce', 'unsubscribe', 'marketing',
    'info', 'hello', 'hi',    # generic blasts (tune to taste)
}

# Known email marketing infrastructure domains
MARKETING_INFRA_DOMAINS = {
    'mailchimp.com', 'list-manage.com', 'campaign-archive.com',
    'sendgrid.net', 'sendgrid.com',
    'constantcontact.com', 'r.constantcontact.com',
    'klaviyo.com', 'em.klaviyo.com',
    'hubspot.com', 'hs-email.com', 'hubspotemail.net',
    'marketo.com', 'mktomail.com',
    'salesforce.com', 'exacttarget.com',
    'mailgun.org', 'mailgun.net',
    'postmarkapp.com',
    'amazonses.com',
    'responsys.net',
}

# Job board / aggregator notification domains (not personal recruiter outreach)
JOB_BOARD_DOMAINS = {
    'linkedin.com', 'e.linkedin.com', 'em.linkedin.com',
    'indeed.com', 'glassdoor.com', 'ziprecruiter.com',
    'monster.com', 'careerbuilder.com', 'dice.com',
    'simplyhired.com', 'ladders.com', 'idealist.org',
    'usajobs.gov', 'clearancejobs.com',
}

# Domains treated as personal (excluded from all tables)
PERSONAL_DOMAINS = {
    'yahoo.com', 'ymail.com',
    'gmail.com', 'googlemail.com',
    'hotmail.com', 'hotmail.co.uk',
    'outlook.com', 'live.com', 'msn.com',
    'aol.com',
    'icloud.com', 'me.com', 'mac.com',
    'protonmail.com', 'proton.me',
    'comcast.net', 'att.net', 'verizon.net',
    'sbcglobal.net', 'bellsouth.net',
    'zoho.com',
}

print('Config ready ✓')

In [ ]:
# ── Create / Connect SQLite DB ────────────────────────────────────────────────
con = sqlite3.connect(DB_PATH)
cur = con.cursor()

# raw_emails — every email from IMAP, untouched
cur.execute('''
    CREATE TABLE IF NOT EXISTS raw_emails (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        msg_id      TEXT    UNIQUE,
        from_name   TEXT,
        from_email  TEXT,
        domain      TEXT,
        subject     TEXT,
        date_sent   TEXT
    )
''')

# staging — deduplicated (one row per from_email), with category for review
cur.execute('''
    CREATE TABLE IF NOT EXISTS staging (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        from_name       TEXT,
        from_email      TEXT    UNIQUE,
        domain          TEXT,
        email_count     INTEGER DEFAULT 1,
        first_contact   TEXT,
        last_contact    TEXT,
        category        TEXT    DEFAULT "unknown",
        auto_flagged    INTEGER DEFAULT 0,
        notes           TEXT
    )
''')
# category values: unknown | recruiter | marketing | newsletter | notification
# auto_flagged: 1 = set by rules, 0 = set manually

# recruiters — final clean list, promoted from staging
cur.execute('''
    CREATE TABLE IF NOT EXISTS recruiters (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        from_name       TEXT,
        from_email      TEXT    UNIQUE,
        domain          TEXT,
        email_count     INTEGER DEFAULT 1,
        first_contact   TEXT,
        last_contact    TEXT,
        outreach_sent   INTEGER DEFAULT 0,
        outreach_date   TEXT
    )
''')

con.commit()
print('Database & tables ready ✓  (raw_emails → staging → recruiters)')

In [ ]:
# ── Connect to Yahoo IMAP ─────────────────────────────────────────────────────
imap = imaplib.IMAP4_SSL('imap.mail.yahoo.com')
imap.login(YAHOO_EMAIL, YMAIL_PW)
print('IMAP connected ✓')

In [ ]:
# ── Batch Fetch Headers (avoids Yahoo rate limiting) ──────────────────────────
# 150 emails per IMAP request → ~67 total requests vs 10,000

imap.select('INBOX', readonly=True)
status, messages = imap.search(None, 'ALL')
email_ids    = messages[0].split()
total_emails = len(email_ids)
print(f'Emails found in INBOX: {total_emails}')

FETCH_CMD  = '(BODY.PEEK[HEADER.FIELDS (FROM DATE SUBJECT MESSAGE-ID)])'
BATCH_SIZE = 150
records    = []
errors     = 0

def parse_header_item(item, fallback_id=''):
    try:
        msg        = email.message_from_bytes(item)
        from_raw   = msg.get('From', '')
        name, addr = parseaddr(from_raw)
        addr       = addr.lower().strip()
        if not addr or '@' not in addr:
            return None
        domain  = addr.split('@')[-1]
        msg_id  = msg.get('Message-ID', f'no-id-{fallback_id}').strip()
        subject = msg.get('Subject', '').strip()
        try:
            date_sent = parsedate_to_datetime(msg.get('Date', '')).isoformat()
        except Exception:
            date_sent = None
        return {
            'msg_id':     msg_id,
            'from_name':  name.strip(),
            'from_email': addr,
            'domain':     domain,
            'subject':    subject,
            'date_sent':  date_sent,
        }
    except Exception:
        return None

for batch_start in range(0, total_emails, BATCH_SIZE):
    batch   = email_ids[batch_start : batch_start + BATCH_SIZE]
    ids_str = b','.join(batch)
    try:
        status, data = imap.fetch(ids_str, FETCH_CMD)
        for item in data:
            if isinstance(item, tuple) and len(item) == 2:
                rec = parse_header_item(item[1], fallback_id=batch_start)
                if rec:
                    records.append(rec)
    except Exception as e:
        errors += len(batch)
        print(f'  Batch error at {batch_start}: {e}')

    processed = min(batch_start + BATCH_SIZE, total_emails)
    if processed % 1000 == 0 or processed == total_emails:
        print(f'  Fetched {processed} / {total_emails} ...')

imap.logout()
print(f'\nDone. Parsed {len(records)} emails | Errors: {errors}')

In [ ]:
# ── Insert raw_emails (skip duplicates) ───────────────────────────────────────
cur.executemany('''
    INSERT OR IGNORE INTO raw_emails
        (msg_id, from_name, from_email, domain, subject, date_sent)
    VALUES
        (:msg_id, :from_name, :from_email, :domain, :subject, :date_sent)
''', records)
con.commit()
total = cur.execute('SELECT COUNT(*) FROM raw_emails').fetchone()[0]
print(f'raw_emails: {total} rows')

In [ ]:
# ── Build Staging: Deduplicate + Auto-Categorize ───────────────────────────────
# One row per from_email. Auto-categorization runs first pass; review in next cells.

df = pd.read_sql('SELECT * FROM raw_emails', con)

# Deduplicate — one row per sender, exclude personal domains
staged = (
    df[~df['domain'].isin(PERSONAL_DOMAINS)]
    .dropna(subset=['from_email'])
    .groupby('from_email', as_index=False)
    .agg(
        from_name     = ('from_name',  'first'),
        domain        = ('domain',     'first'),
        email_count   = ('msg_id',     'count'),
        first_contact = ('date_sent',  'min'),
        last_contact  = ('date_sent',  'max'),
    )
)

def auto_categorize(row):
    """Return (category, auto_flagged). Rules run in priority order."""
    addr   = row['from_email'].lower()
    domain = row['domain'].lower()
    local  = addr.split('@')[0]  # part before @

    # 1. Job board notifications
    if domain in JOB_BOARD_DOMAINS:
        return 'notification', 1

    # 2. Known marketing infra
    if domain in MARKETING_INFRA_DOMAINS:
        return 'marketing', 1

    # 3. noreply / newsletter patterns in local address
    if any(p in local for p in NOREPLY_PATTERNS):
        return 'marketing', 1

    # 4. High volume — recruiter wouldn't email you 20+ times
    if row['email_count'] > MARKETING_THRESHOLD:
        return 'marketing', 1

    return 'unknown', 0

staged[['category', 'auto_flagged']] = staged.apply(
    auto_categorize, axis=1, result_type='expand'
)

# Summary before writing to DB
print('Auto-categorization result:')
print(staged['category'].value_counts().to_string())
print(f'\nTotal unique senders in staging: {len(staged)}')

In [ ]:
# ── Upsert into staging (preserve manual edits on re-run) ────────────────────
# ON CONFLICT: refresh email_count and contacts, but DON'T overwrite category
# if it was manually set (auto_flagged = 0 means human touched it)

for _, row in staged.iterrows():
    cur.execute('''
        INSERT INTO staging
            (from_name, from_email, domain, email_count,
             first_contact, last_contact, category, auto_flagged)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(from_email) DO UPDATE SET
            email_count   = excluded.email_count,
            last_contact  = excluded.last_contact,
            first_contact = excluded.first_contact,
            -- Only overwrite category if it was auto-set, not manually edited
            category      = CASE WHEN staging.auto_flagged = 1
                                 THEN excluded.category
                                 ELSE staging.category END,
            auto_flagged  = CASE WHEN staging.auto_flagged = 1
                                 THEN excluded.auto_flagged
                                 ELSE staging.auto_flagged END
    ''', (
        row['from_name'], row['from_email'], row['domain'],
        row['email_count'], row['first_contact'], row['last_contact'],
        row['category'], row['auto_flagged']
    ))

con.commit()
total = cur.execute('SELECT COUNT(*) FROM staging').fetchone()[0]
print(f'staging: {total} rows')

In [ ]:
# ── Review: Unknowns to Categorize ───────────────────────────────────────────
# These need your eyes. High email_count = probably still marketing.
# Low count from a company domain = likely a recruiter.

unknowns = pd.read_sql('''
    SELECT from_name, from_email, domain, email_count, last_contact
    FROM   staging
    WHERE  category = "unknown"
    ORDER  BY email_count DESC
''', con)

print(f'Unknowns to review: {len(unknowns)}')
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.max_rows', 60)
unknowns

In [ ]:
# ── Manual Overrides ──────────────────────────────────────────────────────────
# Run the block(s) you need, then re-run cell_11 to refresh recruiters.
# Setting auto_flagged=0 protects these from being overwritten on re-runs.

def set_category(emails_or_domains, category, by='email'):
    """Bulk-update category. by='email' or by='domain'."""
    if isinstance(emails_or_domains, str):
        emails_or_domains = [emails_or_domains]
    col = 'from_email' if by == 'email' else 'domain'
    placeholders = ','.join('?' * len(emails_or_domains))
    cur.execute(f'''
        UPDATE staging
        SET    category = ?, auto_flagged = 0
        WHERE  {col} IN ({placeholders})
    ''', [category] + list(emails_or_domains))
    con.commit()
    print(f'Updated {cur.rowcount} row(s) → {category}')

# ── Examples (uncomment and edit as needed) ───────────────────────────────────

# Mark a single email as recruiter:
# set_category('recruiter@somecompany.com', 'recruiter')

# Mark an entire domain as marketing:
# set_category('someretailer.com', 'marketing', by='domain')

# Bulk-mark all unknowns with count > 10 as marketing:
# cur.execute('''
#     UPDATE staging SET category = 'marketing', auto_flagged = 0
#     WHERE  category = 'unknown' AND email_count > 10
# ''')
# con.commit(); print(f'Marked {cur.rowcount} rows as marketing')

# Mark a list of recruiter emails at once:
# set_category([
#     'jane@recruitingfirm.com',
#     'bob@staffingco.com',
# ], 'recruiter')

print('Override helpers ready — uncomment the block(s) you need')

In [ ]:
# ── Promote staging → recruiters (only confirmed recruiters) ──────────────────
# Run this after reviewing / overriding categories in the cell above.

recruiter_rows = pd.read_sql('''
    SELECT from_name, from_email, domain, email_count, first_contact, last_contact
    FROM   staging
    WHERE  category = "recruiter"
''', con)

for _, row in recruiter_rows.iterrows():
    cur.execute('''
        INSERT INTO recruiters
            (from_name, from_email, domain, email_count, first_contact, last_contact)
        VALUES (?, ?, ?, ?, ?, ?)
        ON CONFLICT(from_email) DO UPDATE SET
            email_count   = excluded.email_count,
            last_contact  = excluded.last_contact,
            first_contact = excluded.first_contact
    ''', (
        row['from_name'], row['from_email'], row['domain'],
        row['email_count'], row['first_contact'], row['last_contact']
    ))

con.commit()
total = cur.execute('SELECT COUNT(*) FROM recruiters').fetchone()[0]
print(f'recruiters table: {total} confirmed recruiters')

In [ ]:
# ── Stats ─────────────────────────────────────────────────────────────────────
print('=== Staging Category Breakdown ===')
breakdown = pd.read_sql('''
    SELECT   category, COUNT(*) AS count
    FROM     staging
    GROUP BY category
    ORDER BY count DESC
''', con)
print(breakdown.to_string(index=False))

print('\n=== Top 15 Recruiter Domains ===')
top = pd.read_sql('''
    SELECT   domain, COUNT(*) AS senders
    FROM     recruiters
    GROUP BY domain
    ORDER BY senders DESC
    LIMIT    15
''', con)
print(top.to_string(index=False))

print('\n=== Outreach Status ===')
out = pd.read_sql('''
    SELECT
        SUM(CASE WHEN outreach_sent = 0 THEN 1 ELSE 0 END) AS to_contact,
        SUM(CASE WHEN outreach_sent = 1 THEN 1 ELSE 0 END) AS contacted
    FROM recruiters
''', con)
print(out.to_string(index=False))

In [ ]:
# ── Mark Outreach Sent ────────────────────────────────────────────────────────

# email_to_mark = 'recruiter@somecompany.com'
# cur.execute('''
#     UPDATE recruiters
#     SET outreach_sent = 1, outreach_date = DATE('now')
#     WHERE from_email = ?
# ''', (email_to_mark,))
# con.commit()
# print(f'Marked {email_to_mark} as contacted')

print('Outreach cell ready — uncomment and set email_to_mark to use')

In [ ]:
# ── Export confirmed recruiters to CSV ────────────────────────────────────────
export_df = pd.read_sql('''
    SELECT from_name, from_email, domain, email_count,
           first_contact, last_contact, outreach_sent, outreach_date
    FROM   recruiters
    WHERE  outreach_sent = 0
    ORDER  BY last_contact DESC
''', con)

export_df.to_csv('recruiters_to_contact.csv', index=False)
print(f'Exported {len(export_df)} recruiters → recruiters_to_contact.csv')

con.close()